# Comparing environment-aware configuration: typed settings vs layered loaders vs simple env readers

## Purpose

Services that move between local development, CI, and deployed environments need configuration that changes without code edits. This notebook compares three widely used Python approaches side by side: typed settings classes (the Pydantic settings pattern), layered multi-source loaders (the dynaconf pattern), and minimal typed environment readers (the python-decouple pattern). Each pattern is shown with runnable code so the trade-offs are visible in behavior, not just prose.

## When to use each approach

- **Typed settings classes** suit services with many related settings or validation rules. Configuration becomes an object with named fields and type coercion, so a missing or malformed value fails at load time with a clear message.
- **Layered multi-source loaders** suit projects that combine several sources with a defined precedence (per-environment files, shared defaults, environment overrides, secrets). One loader merges them so call sites read a single view.
- **Minimal typed environment readers** suit small tools and scripts where the only requirement is reading a dozen flat values from the environment or a plain env file, with light casting and defaults.

Rule of thumb: one flat group of values favors the minimal reader; validation-heavy services favor typed settings; multi-environment file layering favors the layered loader.

## Prerequisites

- A Python interpreter with the standard library only (every cell below runs on the standard library; blocks that name third-party packages degrade gracefully when the package is absent).
- An environment where process environment variables can be set (local shell or CI step).
- Familiarity with exporting variables in a shell and with keeping secrets out of version control.

In [ ]:
import os

# last_verified: 2026-09-17 · python n/a
# Baseline: plain standard-library environment reading with explicit defaults.
# This is the behavior every higher-level library builds on.

os.environ.setdefault("APP_LOG_LEVEL", "info")
os.environ.setdefault("APP_RETRIES", "3")

log_level = os.environ.get("APP_LOG_LEVEL", "info")
try:
    retries = int(os.environ.get("APP_RETRIES", "3"))
except ValueError as exc:
    raise SystemExit(f"invalid APP_RETRIES value: {os.environ.get('APP_RETRIES')!r}") from exc

print(f"log_level={log_level} retries={retries}")

## Steps

1. Read the baseline above and note the two chores every approach must handle: sourcing values and validating them.
2. Run the typed-settings cell and observe load-time validation: bad input raises before the program starts its main logic.
3. Run the layered-loader cell and observe precedence: an environment value wins over a file value, which wins over a built-in default.
4. Run the minimal-reader cell and observe simplicity: one helper, one cast, one default per setting.
5. Compare the three against the service at hand using the decision table in the final markdown cell.

In [ ]:
from dataclasses import dataclass, field

# Pattern 1: typed settings class (the Pydantic settings pattern).
# With the third-party package installed this would be a BaseSettings subclass;
# the standard-library equivalent below shows the same contract: coerce,
# validate at construction, fail fast with a clear message.

try:
    from pydantic_settings import BaseSettings  # type: ignore
    HAS_PYDANTIC_SETTINGS = True
except ImportError:
    HAS_PYDANTIC_SETTINGS = False

print(f"pydantic-settings importable: {HAS_PYDANTIC_SETTINGS}")


@dataclass
class ServiceSettings:
    """Typed settings with load-time validation."""

    log_level: str = "info"
    retries: int = 3
    endpoint: str = ""

    def __post_init__(self) -> None:
        allowed = {"debug", "info", "warning", "error"}
        if self.log_level not in allowed:
            raise ValueError(f"log_level must be one of {sorted(allowed)}")
        if self.retries < 0:
            raise ValueError("retries must be zero or positive")
        if not self.endpoint:
            raise ValueError("endpoint is required")


settings = ServiceSettings(
    log_level=os.environ.get("APP_LOG_LEVEL", "info"),
    retries=int(os.environ.get("APP_RETRIES", "3")),
    endpoint=os.environ.get("APP_ENDPOINT", "api.example.internal"),
)
print(settings)

# Demonstrate fail-fast: an invalid value raises at construction, not mid-run.
try:
    ServiceSettings(log_level="verbose", retries=3, endpoint="x")
except ValueError as exc:
    print(f"rejected as expected: {exc}")

In [ ]:
# Pattern 2: layered multi-source loader (the dynaconf pattern).
# Real deployments merge per-environment files, shared defaults, environment
# overrides, and secrets with a defined precedence. The merge below uses plain
# dicts so the precedence rule is visible and runnable without extra packages.

try:
    import dynaconf  # type: ignore
    HAS_DYNACONF = True
except ImportError:
    HAS_DYNACONF = False

print(f"dynaconf importable: {HAS_DYNACONF}")

defaults = {"log_level": "info", "retries": 3, "endpoint": "api.example.internal"}
file_layer = {"log_level": "debug", "retries": 3}  # stands in for settings file
env_layer = {}
if "APP_LOG_LEVEL" in os.environ:
    env_layer["log_level"] = os.environ["APP_LOG_LEVEL"]
if "APP_RETRIES" in os.environ:
    env_layer["retries"] = int(os.environ["APP_RETRIES"])

merged = {**defaults, **file_layer, **env_layer}
print(f"defaults : {defaults}")
print(f"file     : {file_layer}")
print(f"env      : {env_layer}")
print(f"merged   : {merged}")
assert merged["log_level"] == env_layer.get("log_level", file_layer.get("log_level", defaults["log_level"]))

In [ ]:
# Pattern 3: minimal typed environment reader (the python-decouple pattern).
# One helper per value: read, cast, fall back to a default. Ideal for flat
# settings; awkward once values nest or need cross-field validation.

try:
    from decouple import config as decouple_config  # type: ignore
    HAS_DECOUPLE = True
except ImportError:
    HAS_DECOUPLE = False

print(f"python-decouple importable: {HAS_DECOUPLE}")


def env(key: str, default: str = "", cast=str):
    """Read one flat setting with a cast and a default."""
    raw = os.environ.get(key, default)
    try:
        return cast(raw)
    except (ValueError, TypeError) as exc:
        raise SystemExit(f"invalid {key} value: {raw!r}") from exc


log_level = env("APP_LOG_LEVEL", default="info")
retries = env("APP_RETRIES", default="3", cast=int)
print(f"log_level={log_level} retries={retries}")

## Verify

Re-run every cell top to bottom in a fresh kernel after exporting an override (for example setting the log level variable to a different valid value). Expected result: each pattern reports the override, the typed-settings cell still rejects an invalid level at construction, and the layered cell shows the environment layer winning over the file layer. If any cell raises unexpectedly, the failure is in the notebook edits, not in the installed packages, because every cell runs on the standard library by default.

## Decision table

| Situation | Preferred pattern | Reason |
|---|---|---|
| Few flat values, script or single job | Minimal typed environment reader | Smallest surface; one helper per value |
| Many related settings, validation matters | Typed settings class | Coercion plus load-time errors in one place |
| Several environments sharing defaults with per-environment files | Layered multi-source loader | Single merged view with explicit precedence |
| Nested structures with comments checked into the repo | Configuration file plus a typed wrapper | File holds grouped defaults; wrapper validates |
| Secrets | Environment injection or the platform secret store, regardless of pattern | Keeps credentials out of files and history |

## Common errors

- **Silent string where a number was expected.** The minimal-reader and baseline cells cast explicitly so a non-numeric retry value fails loudly instead of being compared as text.
- **Required value missing in one environment only.** The typed-settings cell raises on an empty endpoint at construction; add the variable to the sample env file so reviewers notice the gap.
- **File value shadowing an environment override.** The layered cell merges in defaults, file, environment order; reversing the merge order reintroduces the shadowing, so keep environment last.

## References

- The companion comparison doc under python/docs covering configuration surfaces in general.
- The reusable config-loader script under python/scripts showing the typed-settings pattern in a deployable file.